In [2]:
import duckdb

DAILY_DB = "/home/newt/quant_project/duckdb/daily_data.duckdb"
FACTOR_DB = "/home/newt/quant_project/duckdb/factor_data.duckdb"

conn = duckdb.connect(DAILY_DB)
cols = [row[1] for row in conn.execute("PRAGMA table_info('daily_data')").fetchall()]
assert "date" in cols and "symbol" in cols, f"missing date/symbol: {cols}"

alpha_cols = [c for c in cols if c.startswith("alpha")]
assert alpha_cols, "no alpha* columns found"

print(f"daily_data columns ({len(cols)}): {cols}")
print(f"will move ({len(alpha_cols)}): {alpha_cols}")
n_daily = conn.execute("SELECT COUNT(*) FROM daily_data").fetchone()[0]
print(f"rows: {n_daily:,}")


daily_data columns (112): ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'amount', 'return_day', 'turnover', 'market_value_million', 'log_return_day', 'numtrades', 'limit_status', 'spread', 'next_open_ref', 'next_limit_up', 'next_limit_down', 'attention_stock', 'disposed_stock', 'full_delivery', 'market', 'untradable', 'avoid_early', 'adj_factor', 'foreign_net_buy_lot', 'it_net_buy_lot', 'inst_net_buy_lot', 'margin_net_buy_lot', 'cash_net_buy_lot', 'foreign_net_buy_days', 'it_net_buy_days', 'dealer_net_buy_days', 'inst_net_buy_days', 'margin_net_buy_days', 'cash_net_buy_days', 'foreign_streak_return', 'it_streak_return', 'dealer_streak_return', 'inst_streak_return', 'margin_streak_return', 'cash_streak_return', 'foreign_streak_cum_lot', 'it_streak_cum_lot', 'dealer_streak_cum_lot', 'inst_streak_cum_lot', 'margin_streak_cum_lot', 'cash_streak_cum_lot', 'foreign_streak_cum_amt_thousand', 'it_streak_cum_amt_thousand', 'dealer_streak_cum_amt_thousand', 'inst_streak_cum_amt_th

In [3]:
# attach factor DB and copy date/symbol + alpha*
quote = lambda name: '"' + name.replace('"', '""') + '"'
select_cols = ", ".join(quote(c) for c in ["date", "symbol"] + alpha_cols)

conn.execute(f"ATTACH '{FACTOR_DB}' AS factor")
conn.execute(
    f"CREATE OR REPLACE TABLE factor.factor_data AS SELECT {select_cols} FROM daily_data"
)
n_factor = conn.execute("SELECT COUNT(*) FROM factor.factor_data").fetchone()[0]
print(f"factor_data rows: {n_factor:,}")
assert n_factor == n_daily, f"row mismatch: daily={n_daily:,} factor={n_factor:,}"
display(conn.execute("SELECT * FROM factor.factor_data LIMIT 5").fetchdf())


factor_data rows: 4,708,990


,date,symbol,alpha101_005,alpha101_012,alpha101_018,alpha101_042,alpha101_057,alpha101_073,alpha101_084,alpha191_008,alpha191_026,alpha191_054,alpha191_065
0,2015-06-01,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-06-02,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-06-03,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-06-04,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-06-05,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# drop alpha* from daily_data via EXCLUDE rebuild
exclude_list = ", ".join(quote(c) for c in alpha_cols)
conn.execute("BEGIN TRANSACTION")
try:
    conn.execute(
        f"CREATE TABLE daily_data_new AS SELECT * EXCLUDE ({exclude_list}) FROM daily_data"
    )
    conn.execute("DROP TABLE daily_data")
    conn.execute("ALTER TABLE daily_data_new RENAME TO daily_data")
    conn.execute("COMMIT")
except Exception:
    conn.execute("ROLLBACK")
    raise

conn.execute("CHECKPOINT")
remaining = [row[1] for row in conn.execute("PRAGMA table_info('daily_data')").fetchall()]
print(f"daily_data after drop ({len(remaining)}): {remaining}")
assert not any(c.startswith("alpha") for c in remaining)


daily_data after drop (101): ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'amount', 'return_day', 'turnover', 'market_value_million', 'log_return_day', 'numtrades', 'limit_status', 'spread', 'next_open_ref', 'next_limit_up', 'next_limit_down', 'attention_stock', 'disposed_stock', 'full_delivery', 'market', 'untradable', 'avoid_early', 'adj_factor', 'foreign_net_buy_lot', 'it_net_buy_lot', 'inst_net_buy_lot', 'margin_net_buy_lot', 'cash_net_buy_lot', 'foreign_net_buy_days', 'it_net_buy_days', 'dealer_net_buy_days', 'inst_net_buy_days', 'margin_net_buy_days', 'cash_net_buy_days', 'foreign_streak_return', 'it_streak_return', 'dealer_streak_return', 'inst_streak_return', 'margin_streak_return', 'cash_streak_return', 'foreign_streak_cum_lot', 'it_streak_cum_lot', 'dealer_streak_cum_lot', 'inst_streak_cum_lot', 'margin_streak_cum_lot', 'cash_streak_cum_lot', 'foreign_streak_cum_amt_thousand', 'it_streak_cum_amt_thousand', 'dealer_streak_cum_amt_thousand', 'inst_streak_cum_amt

In [5]:
# self-check
daily_cols = [row[1] for row in conn.execute("PRAGMA table_info('daily_data')").fetchall()]
factor_cols = [row[1] for row in conn.execute("PRAGMA table_info('factor.factor_data')").fetchall()]
n_daily_final = conn.execute("SELECT COUNT(*) FROM daily_data").fetchone()[0]
n_factor_final = conn.execute("SELECT COUNT(*) FROM factor.factor_data").fetchone()[0]

assert not any(c.startswith("alpha") for c in daily_cols), daily_cols
assert set(factor_cols) == set(["date", "symbol"] + alpha_cols), factor_cols
assert n_daily_final == n_factor_final == n_daily, (n_daily_final, n_factor_final, n_daily)

print("OK")
print(f"  daily_data cols: {daily_cols}")
print(f"  factor_data cols: {factor_cols}")
print(f"  rows: {n_daily_final:,}")

conn.close()


OK
  daily_data cols: ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'amount', 'return_day', 'turnover', 'market_value_million', 'log_return_day', 'numtrades', 'limit_status', 'spread', 'next_open_ref', 'next_limit_up', 'next_limit_down', 'attention_stock', 'disposed_stock', 'full_delivery', 'market', 'untradable', 'avoid_early', 'adj_factor', 'foreign_net_buy_lot', 'it_net_buy_lot', 'inst_net_buy_lot', 'margin_net_buy_lot', 'cash_net_buy_lot', 'foreign_net_buy_days', 'it_net_buy_days', 'dealer_net_buy_days', 'inst_net_buy_days', 'margin_net_buy_days', 'cash_net_buy_days', 'foreign_streak_return', 'it_streak_return', 'dealer_streak_return', 'inst_streak_return', 'margin_streak_return', 'cash_streak_return', 'foreign_streak_cum_lot', 'it_streak_cum_lot', 'dealer_streak_cum_lot', 'inst_streak_cum_lot', 'margin_streak_cum_lot', 'cash_streak_cum_lot', 'foreign_streak_cum_amt_thousand', 'it_streak_cum_amt_thousand', 'dealer_streak_cum_amt_thousand', 'inst_streak_cum_amt_thousa